# GTEx primary LCM README: assay scope and sample counts


GT-HOST-TX-01: retrieve only the primary README_LCM_pilot.txt object discovered in the public v11 object listing; verify its provider MD5/size. No per-sample data are downloaded. Published assay text is retained externally; the notebook stores its receipt, not the paper text.

In [1]:
import json, hashlib, base64, urllib.parse
from pathlib import Path
from scripts.recover_source_files import download
base=Path('results/host_expression_source_review_2026_10_08')
metadata=json.loads((base/'gtex_v11_objects.json').read_text())
items=[i for i in metadata['items'] if Path(i['name']).name=='README_LCM_pilot.txt']
assert len(items)==1 and items[0]['bucket']=='adult-gtex'
item=items[0]
url='https://storage.googleapis.com/adult-gtex/'+urllib.parse.quote(item['name'],safe='/')
target=Path('/media/carruthers/mnt3/claude/toxoplasma_projects/datasets/host_expression_review_inputs/GTEx_v11/README_LCM_pilot.txt')
receipt={'url':url,'path':str(target),'status':'unavailable_continue'}
try:
    if not target.exists(): download(url,target,limit=1<<20)
    data=target.read_bytes()
    assert len(data)==int(item['size'])
    assert base64.b64encode(hashlib.md5(data).digest()).decode()==item['md5Hash']
    data.decode('utf-8')
    receipt.update(status='primary_object_MD5_size_verified',sha256=hashlib.sha256(data).hexdigest(),object_metadata=item)
except Exception as error:
    receipt['gap']=type(error).__name__+': '+str(error)
(base/'lcm_readme_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
print({k:receipt[k] for k in ('status','path','url')})

{'status': 'primary_object_MD5_size_verified', 'path': '/media/carruthers/mnt3/claude/toxoplasma_projects/datasets/host_expression_review_inputs/GTEx_v11/README_LCM_pilot.txt', 'url': 'https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/README_LCM_pilot.txt'}
